# Innovation and Entrepreneurship — Three Reasons the Parts Do Not Sum to the Whole

Business Dynamics Statistics is the cleanest source in this corpus and the easiest to
aggregate wrongly. Its published values are internally exact to the unit. Its sums are
almost never what a reader expects, for **three unrelated reasons that act in opposite
directions** — and the one analysts are trained to worry about, disclosure-avoidance
perturbation, turns out to be the smallest of the three by a factor of several hundred.

There is a fourth thing going on, and it is newer than the data. On **4 June 2026** the
Department of Commerce issued Administrative Order **DAO 216-26**, whose §5.01(c) reads:
*"Noise infusion shall not be used for any statistical product."* Noise infusion is exactly
how every number in this notebook was protected. So the protection method is not a fixed
property of the source. It is a **dated administrative variable**, and a series that spans
the change is not homogeneous in a way that nothing in the values will announce.

This notebook measures all four, and builds a test that a future reader can re-run to
detect which regime produced a vintage they were handed.

## Learning Objectives

1. **Distinguish three failure modes of aggregation** — definitional double counting,
   coverage exclusion, and injected noise — and tell them apart from the data alone.
2. **Read a disclosure-avoidance method as policy**, with an issue date and a successor,
   rather than as a fixed characteristic of a dataset.
3. **Build a regime detector** from the digit distribution of published values.
4. **Recognise a withheld cell that arrives as the number zero**, and see what asking for
   the flag column changes and what it does not.
5. **Reconstruct a published rate**, and learn why its denominator must be derived inside
   one vintage rather than read from last year's file.

## Prerequisites

- Handbook §1.4 (QCEW — suppression with a flag) and §1.5 (CBP — noise infusion). BDS uses
  **both**, which is why it belongs at the end of that sequence rather than inside it.
- Handbook §5 (the vintage policy). This notebook adds a second axis to it: not only *which
  vintage*, but *which protection regime that vintage was produced under*.

## Data Sources

| endpoint | geography | what a row is |
|---|---|---|
| `api.census.gov/data/timeseries/bds` | nation, state, county | one geography × industry × year cell |

Universe: employer establishments in the Longitudinal Business Database, 1978–2023.
Reference year throughout is **2022**; every figure below is fetched live and stored.

Three documents govern the reading and are quoted where used: the BDS **FAQ** (firm
additivity, the entry-rate denominator, which variables carry noise), the BDS **codebook and
glossary** (the flag values, the statewide 999 county code), and **DAO 216-26** itself.

In [1]:
"""Configuration. Every choice that could drift is named here."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.sources import (
    verify_digest,
    BDS_DISCLOSURE_REGIME,
    BDS_FIRM_LEVEL_VARIABLES,
    BDS_FLAG_MEANINGS,
    BDS_NOISE_INFUSED_VARIABLES,
    bds_prior_year_establishments,
    fetch_bds,
)

YEAR = 2022                 # a published year, named; the client rejects "latest"
FOCUS_STATE = "51"          # Virginia, for the single-state sections
COUNT_VARS = ["ESTAB", "EMP"]
FLOW_VARS = ["JOB_CREATION", "JOB_DESTRUCTION", "NET_JOB_CREATION", "FIRMDEATH_ESTABS"]

pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 30)

print(f"reference year        : {YEAR}")
print(f"disclosure regime     : {BDS_DISCLOSURE_REGIME}")
print(f"firm-level variables  : {sorted(BDS_FIRM_LEVEL_VARIABLES)}")
print(f"noise-infused variables: {sorted(BDS_NOISE_INFUSED_VARIABLES)}")
print(f"flag values honoured  : {BDS_FLAG_MEANINGS}")

#: The digest of the payload every figure below was computed from, on this
#: record's own hash basis. The publisher serves no prior vintage, so this pin
#: does not restore one -- it guarantees an upstream change cannot pass
#: unnoticed: the notebook raises instead of reporting different numbers under
#: unchanged prose. Handbook §5.2.
PINNED_BDS_US = "4d7e477d1914cae34942684edd805687079ce8605d29077109a73fe7904770fc"


reference year        : 2022
disclosure regime     : hybrid balanced multiplicative noise infusion (adapted from County Business Patterns); prohibited prospectively by DoC DAO 216-26 Sec. 5.01(c), effective 2026-06-04, with coarsening preferred and suppression permitted only as a last resort under Sec. 5.02
firm-level variables  : ['FIRM', 'FIRMDEATH_FIRMS']
noise-infused variables: ['JOB_CREATION', 'JOB_CREATION_BIRTHS', 'JOB_CREATION_CONTINUERS', 'JOB_DESTRUCTION', 'JOB_DESTRUCTION_CONTINUERS', 'JOB_DESTRUCTION_DEATHS', 'NET_JOB_CREATION']
flag values honoured  : {'D': 'disclosure suppression -- too few firms in the cell', 'S': 'data-quality suppression -- cell unreliable on time-series grounds', 'X': 'structurally missing or structurally zero', 'N': 'rate not calculable -- denominator is zero'}


In [2]:
"""Retrieve every frame this notebook uses, once."""
us, prov_us = verify_digest(
    fetch_bds(year=YEAR, variables=["ESTAB", "EMP", "FIRM"], geography="us:*"),
    PINNED_BDS_US, label='Census BDS, national 2022')
states, _ = fetch_bds(year=YEAR, variables=["ESTAB", "EMP", "FIRM"], geography="state:*")
counties, prov_cty = fetch_bds(
    year=YEAR, variables=["ESTAB", "EMP", "FIRM"], geography="county:*"
)

va_total, _ = fetch_bds(
    year=YEAR, variables=COUNT_VARS + FLOW_VARS, geography=f"state:{FOCUS_STATE}"
)
va_sectors, _ = fetch_bds(
    year=YEAR, variables=COUNT_VARS + FLOW_VARS,
    geography=f"state:{FOCUS_STATE}", naics="*",
)
va_counties, _ = fetch_bds(
    year=YEAR,
    variables=["ESTAB", "ESTABS_ENTRY", "ESTABS_EXIT", "ESTABS_ENTRY_RATE", "DENOM"],
    geography="county:*", within=f"state:{FOCUS_STATE}",
)

print(f"nation           : {len(us):>7,} rows")
print(f"states           : {len(states):>7,} rows")
print(f"counties         : {len(counties):>7,} rows")
print(f"Virginia sectors : {len(va_sectors):>7,} rows")
print(f"Virginia counties: {len(va_counties):>7,} rows")
print()
for note in prov_cty.notes:
    print(f"  - {note}")

nation           :       1 rows
states           :      51 rows
counties         :   3,144 rows
Virginia sectors :      20 rows
Virginia counties:     133 rows

  - Disclosure-avoidance regime of this vintage: hybrid balanced multiplicative noise infusion (adapted from County Business Patterns); prohibited prospectively by DoC DAO 216-26 Sec. 5.01(c), effective 2026-06-04, with coarsening preferred and suppression permitted only as a last resort under Sec. 5.02.
  - Noise is fixed at tabulation, not per request: repeating a query returns identical values, so the perturbation cannot be averaged away by resampling.
  - NOT ADDITIVE over geography or industry: FIRM. A firm with establishments in several counties, states or sectors is counted once in each, so summing these across a partition double-counts.
  - County tabulations exclude the statewide (999) records, which the codebook assigns to 'establishments without a fixed location such as employee leasing establishments'. The API does 

## 1. The protection method has an issue date

Every published BDS value was protected by *hybrid balanced multiplicative noise infusion*,
adapted from County Business Patterns: a random multiplier is applied to each establishment
record **before** tabulation. DAO 216-26 §5.01(c) now prohibits that method prospectively,
and §5.02 replaces it with **coarsening** ("preferred") and **suppression** ("as a last
resort").

The three methods leave different fingerprints in published values. Noise infusion leaves
the last digit of a large count essentially uniform, because a multiplicative perturbation
of a five- or six-digit number lands anywhere. Coarsening — rounding to a base — does the
opposite: it concentrates the last digits on zero, or on zero and five.

That gives a reader a cheap, reproducible way to ask which regime produced a file they were
handed, without reading a methodology page that may not have been updated. Here is the test
on this vintage.

In [3]:
"""A last-digit test as a disclosure-regime detector."""
emp = counties["EMP"].dropna()
emp = emp[emp > 0].astype("int64")

digits = emp % 10
observed = np.array([int((digits == d).sum()) for d in range(10)], dtype=float)
expected = np.full(10, observed.sum() / 10.0)
chi2, p_value = stats.chisquare(f_obs=observed, f_exp=expected)

table = pd.DataFrame(
    {"last_digit": range(10), "observed": observed.astype(int),
     "share": observed / observed.sum()}
)
print(table.to_string(index=False, formatters={"share": "{:.3f}".format}))
print()
print(f"county employment values tested : {len(emp):,}")
print(f"chi-square (9 df)               : {chi2:.2f}")
print(f"p-value                         : {p_value:.3f}")
print(f"values ending in 0              : {int((digits == 0).sum()):,} "
      f"({(digits == 0).mean():.1%})")
print(f"values divisible by 100         : {int((emp % 100 == 0).sum()):,} "
      f"({(emp % 100 == 0).mean():.1%})")

 last_digit  observed share
          0       341 0.109
          1       333 0.106
          2       294 0.094
          3       309 0.098
          4       309 0.098
          5       302 0.096
          6       326 0.104
          7       308 0.098
          8       325 0.103
          9       295 0.094

county employment values tested : 3,142
chi-square (9 df)               : 7.47
p-value                         : 0.589
values ending in 0              : 341 (10.9%)
values divisible by 100         : 37 (1.2%)


The digits are flat. Under coarsening to any base they could not be: rounding to the nearest
10 would put every value in the `0` row, and rounding to a base of 5 would put them in two.
So the fingerprint of this vintage is *perturbation, not rounding* — consistent with the
method Census documents and with the order that ends it.

**Keep this cell.** It is not a result about 2022; it is an instrument. Re-run it on the
first BDS vintage published after the transition and the same three lines will say which
regime you are holding, whatever the methodology page says. The corresponding caution is
that it detects *rounding*, not *suppression*: DAO 216-26 permits suppression as a last
resort, and suppression leaves the surviving digits exactly as flat as they are here. §4
measures that separately.

## 2. Where the noise actually is

The word "noise-infused" attached to a whole dataset is close to useless, because it does
not say which variables carry it or how much. The Census FAQ is specific:

> "Noise is added at the cell level for certain variables, including: job creation, job
> creation births, job creation continuers, job destruction, job destruction deaths, job
> destruction continuers, and net job creation, this can cause discrepancies in these
> statistics across tables."

That is a testable claim, and the test is clean because a sector decomposition is an exact
partition of the same establishment records within one geography. If the counts are
tabulated from perturbed records, the sums must be exact. If a statistic has noise added to
the published cell, they cannot be.

In [4]:
"""Do the sectors sum to the state? By variable."""
sectors = va_sectors[va_sectors["NAICS"] != "00"]

rows = []
for var in COUNT_VARS + FLOW_VARS:
    total = float(va_total[var].iloc[0])
    summed = float(sectors[var].sum(min_count=1))
    rows.append({
        "variable": var,
        "state_total": total,
        "sum_of_sectors": summed,
        "gap": summed - total,
        "gap_pct": 100 * (summed - total) / total if total else np.nan,
        "census_says_noisy": var in BDS_NOISE_INFUSED_VARIABLES,
    })

additivity = pd.DataFrame(rows)
print(f"Virginia {YEAR}, {len(sectors)} NAICS sectors, one geography")
print(f"suppressed sector cells in this frame: "
      f"{int(sectors[[f'{v}_F' for v in COUNT_VARS + FLOW_VARS]].notna().sum().sum())}")
print()
print(additivity.to_string(index=False, formatters={
    "state_total": "{:,.0f}".format, "sum_of_sectors": "{:,.0f}".format,
    "gap": "{:+,.0f}".format, "gap_pct": "{:+.4f}".format}))

exact = additivity.loc[~additivity["census_says_noisy"], "gap"]
noisy = additivity.loc[additivity["census_says_noisy"], "gap_pct"].abs()
print()
print(f"variables Census does not list as noisy, maximum |gap|: {exact.abs().max():,.0f}")
print(f"variables Census does list as noisy, largest |gap|    : {noisy.max():.4f}%")

Virginia 2022, 19 NAICS sectors, one geography
suppressed sector cells in this frame: 0

        variable state_total sum_of_sectors gap gap_pct  census_says_noisy
           ESTAB     189,635        189,635  +0 +0.0000              False
             EMP   3,493,659      3,493,659  +0 +0.0000              False
    JOB_CREATION     519,386        519,375 -11 -0.0021               True
 JOB_DESTRUCTION     383,823        383,771 -52 -0.0135               True
NET_JOB_CREATION     135,563        135,604 +41 +0.0302               True
FIRMDEATH_ESTABS      11,116         11,116  +0 +0.0000              False

variables Census does not list as noisy, maximum |gap|: 0
variables Census does list as noisy, largest |gap|    : 0.0302%


The counts are exact and the flows are not, exactly as documented. The size of the
discrepancy is the part worth carrying: it is a few tens of jobs in a state with millions,
which is to say **the perturbation is real, documented, and analytically irrelevant at this
geography**. An analyst who spent the afternoon worrying about noise infusion spent it on
the smallest of the three problems in this notebook.

Two consequences follow, and both are more useful than the size.

**Balance is what buys the exactness.** The noise multiplier is applied once, to each
establishment record, before tabulation. Any sum of the same perturbed records is therefore
internally consistent, which is why a sector decomposition adds up. It also means additivity
is *not* a test for perturbation: a perturbed table and an unperturbed one look identical
under that check. §1's digit test exists because this one cannot do the job.

**The noise cannot be averaged away.** It is fixed at tabulation, not drawn per request.

In [5]:
"""Three identical requests."""
signatures = []
for attempt in range(3):
    probe, _ = fetch_bds(
        year=YEAR, variables=["ESTAB", "EMP", "JOB_CREATION"],
        geography=f"state:{FOCUS_STATE}",
    )
    signatures.append(tuple(float(probe[v].iloc[0]) for v in ("ESTAB", "EMP", "JOB_CREATION")))

for attempt, signature in enumerate(signatures, start=1):
    print(f"  request {attempt}: ESTAB={signature[0]:,.0f}  EMP={signature[1]:,.0f}  "
          f"JOB_CREATION={signature[2]:,.0f}")
print()
print(f"distinct responses across {len(signatures)} identical requests: "
      f"{len(set(signatures))}")

  request 1: ESTAB=189,635  EMP=3,493,659  JOB_CREATION=519,386
  request 2: ESTAB=189,635  EMP=3,493,659  JOB_CREATION=519,386
  request 3: ESTAB=189,635  EMP=3,493,659  JOB_CREATION=519,386

distinct responses across 3 identical requests: 1


One. Repeating a query returns the same perturbed value, so there is no sampling
distribution to average over and no number of requests that recovers the truth. This is the
opposite of what people expect from the word *random*, and it is the reason a published
noise-infused figure has to be treated as a fixed unknown-error quantity rather than as a
draw.

## 3. The firm count is not additive anywhere

The largest aggregation error in BDS is not statistical at all. From the Census FAQ:

> "Geography and industry are defined at the establishment level instead of the firm level.
> So firms with multiple establishments operating in different geographies or industries will
> show up in the total firm data for each geography or industry in which they operate at
> least one establishment."

A firm with stores in forty counties is one firm nationally and forty firms when you sum
counties. Nothing in the response marks the rows that double-count, and the resulting total
is a plausible number in the right units.

The size of the error is not a constant. It is a function of how finely the geography cuts
the firms' footprints, which means **the same analysis run at two geographic levels gives two
different answers and neither is flagged**.

In [6]:
"""How badly does FIRM double-count, and does it scale with the partition?"""
national = {v: float(us[v].iloc[0]) for v in ("ESTAB", "EMP", "FIRM")}
state_sums = {v: float(states[v].sum(min_count=1)) for v in ("ESTAB", "EMP", "FIRM")}

print(f"national totals vs the sum of the {len(states)} state rows, {YEAR}")
for var in ("ESTAB", "EMP", "FIRM"):
    gap = state_sums[var] - national[var]
    print(f"  {var:6s} nation={national[var]:>13,.0f}  sum(states)={state_sums[var]:>13,.0f}"
          f"  gap={gap:>+11,.0f}  ({100 * gap / national[var]:+.3f}%)")

county_sum = counties.groupby("state")[["ESTAB", "EMP", "FIRM"]].sum(min_count=1)
county_n = counties.groupby("state").size().rename("n_counties")
parent = states.set_index("state")[["ESTAB", "EMP", "FIRM"]]

compare = county_sum.join(parent, lsuffix="_counties", rsuffix="_state").join(county_n)
for var in ("ESTAB", "EMP", "FIRM"):
    compare[f"{var}_gap_pct"] = (
        100 * (compare[f"{var}_counties"] - compare[f"{var}_state"]) / compare[f"{var}_state"]
    )

multi = compare[compare["n_counties"] > 1]
rho, rho_p = stats.spearmanr(multi["n_counties"], multi["FIRM_gap_pct"])
density = multi["ESTAB_state"] / multi["n_counties"]
rho_density, _ = stats.spearmanr(density, multi["FIRM_gap_pct"])

print()
print(f"summing counties to the state, {len(multi)} states with more than one county")
print(f"  FIRM  gap%: min={multi['FIRM_gap_pct'].min():+.2f}  "
      f"median={multi['FIRM_gap_pct'].median():+.2f}  max={multi['FIRM_gap_pct'].max():+.2f}")
print(f"  states where the county sum EXCEEDS the state FIRM count: "
      f"{int((multi['FIRM_gap_pct'] > 0).sum())}/{len(multi)}")
print()
print(f"  Spearman rho(number of counties, FIRM gap%)          = {rho:+.3f}  (p={rho_p:.2e})")
print(f"  Spearman rho(establishments per county, FIRM gap%)   = {rho_density:+.3f}")
fine = multi.loc[multi["n_counties"] >= 90, "FIRM_gap_pct"]
coarse = multi.loc[multi["n_counties"] <= 20, "FIRM_gap_pct"]
print(f"  median gap, states with >= 90 counties (n={len(fine)})  = {fine.median():+.2f}%")
print(f"  median gap, states with <= 20 counties (n={len(coarse)}) = {coarse.median():+.2f}%")

national totals vs the sum of the 51 state rows, 2022
  ESTAB  nation=    7,408,649  sum(states)=    7,408,649  gap=         +0  (+0.000%)
  EMP    nation=  134,143,137  sum(states)=  134,143,137  gap=         +0  (+0.000%)
  FIRM   nation=    5,541,603  sum(states)=    5,748,177  gap=   +206,574  (+3.728%)

summing counties to the state, 50 states with more than one county
  FIRM  gap%: min=+2.62  median=+10.44  max=+18.79
  states where the county sum EXCEEDS the state FIRM count: 50/50

  Spearman rho(number of counties, FIRM gap%)          = +0.746  (p=4.91e-10)
  Spearman rho(establishments per county, FIRM gap%)   = -0.691
  median gap, states with >= 90 counties (n=12)  = +13.03%
  median gap, states with <= 20 counties (n=10) = +4.84%


The double-count rises with the fineness of the cut, in the direction and with the strength
you would predict if the mechanism is exactly what Census says it is. States carved into
many small counties give every multi-unit firm more boundaries to cross.

There is no correction available here. Recovering a firm count for an arbitrary union of
counties requires knowing which establishments belong to the same firm, which is precisely
what the microdata withholds. The honest handling is to **use ESTAB when you are aggregating
geography, and to quote FIRM only at the geography Census published it for.**

## 4. The deficit, and how to tell exclusion from suppression

The count variables were exact when we cut by industry. They are not exact when we cut by
county, and the direction flips: the counties fall *short* of the state.

In [7]:
"""Two deficits with the same sign and different causes."""
print(f"summing counties to the state, {YEAR}")
for var in ("ESTAB", "EMP"):
    col = f"{var}_gap_pct"
    print(f"  {var:5s} min={multi[col].min():+.2f}%  median={multi[col].median():+.2f}%  "
          f"max={multi[col].max():+.2f}%  negative in "
          f"{int((multi[col] < 0).sum())}/{len(multi)} states")

sign_p = stats.binomtest(
    int((multi["ESTAB_gap_pct"] < 0).sum()), len(multi), 0.5, alternative="two-sided"
).pvalue
print()
print(f"  sign test on the establishment gap: p = {sign_p:.2e}")
print(f"  county rows returned nationwide   : {len(counties):,}")

summing counties to the state, 2022
  ESTAB min=-1.69%  median=-0.71%  max=-0.18%  negative in 50/50 states
  EMP   min=-12.08%  median=-3.43%  max=-0.94%  negative in 50/50 states

  sign test on the establishment gap: p = 1.78e-15
  county rows returned nationwide   : 3,144


A one-sided deficit in fifty states out of fifty is not perturbation. Balanced noise is
centred; this is not centred, and it is not small.

The BDS codebook names the cause:

> "The BDS also includes statewide (999) county codes. Statewide county codes are assigned
> to establishments without a fixed location such as employee leasing establishments."

Those records exist in the published files and the API does not serve them: `county:*`
returns only real counties, and asking for `county:999` is rejected outright. So **the
county-level API data cannot be reconciled to the state total from the API at any effort.**

If that explanation is right it makes a sharp prediction. Employee-leasing establishments —
professional employer organisations — are a small number of records carrying an enormous
number of employees. So the deficit should be *tiny in establishments and huge in
employment*, and it should concentrate in NAICS 56, the sector that contains them. A
competing explanation, cell suppression at fine geography, predicts something different:
suppression removes a whole row at once, so it should shrink establishments and employment
roughly together.

Those two predictions are distinguishable from the data. The ratio of the employment gap to
the establishment gap separates them.

In [8]:
"""Fetch the sector decomposition at both geographic levels."""
state_sectors, _ = fetch_bds(
    year=YEAR, variables=["ESTAB", "EMP"], geography="state:*", naics="*"
)
county_sectors, _ = fetch_bds(
    year=YEAR, variables=["ESTAB", "EMP"], geography="county:*", naics="*"
)
state_sectors = state_sectors[state_sectors["NAICS"] != "00"]
county_sectors = county_sectors[county_sectors["NAICS"] != "00"]

flagged = county_sectors["EMP_F"].notna()
print(f"county x sector cells : {len(county_sectors):,}")
print(f"  suppressed (flag D) : {int(flagged.sum()):,} ({flagged.mean():.2%})")
print(f"  flag values present : {sorted(county_sectors['EMP_F'].dropna().unique())}")
print(f"state x sector cells  : {len(state_sectors):,}  "
      f"suppressed: {int(state_sectors['EMP_F'].notna().sum())}")

county x sector cells : 59,736
  suppressed (flag D) : 7,568 (12.67%)
  flag values present : ['D']
state x sector cells  : 969  suppressed: 1


In [9]:
"""The diagnostic: employment gap relative to establishment gap, by sector."""
parent_sec = state_sectors.groupby("NAICS")[["ESTAB", "EMP"]].sum(min_count=1)
child_sec = county_sectors.groupby("NAICS")[["ESTAB", "EMP"]].sum(min_count=1)
suppressed_sec = (
    county_sectors[county_sectors["EMP_F"].notna()].groupby("NAICS").size()
    .reindex(parent_sec.index, fill_value=0).rename("suppressed_cells")
)

sec = parent_sec.join(child_sec, lsuffix="_state", rsuffix="_county").join(suppressed_sec)
sec["emp_gap"] = sec["EMP_county"] - sec["EMP_state"]
sec["emp_gap_pct"] = 100 * sec["emp_gap"] / sec["EMP_state"]
sec["estab_gap_pct"] = 100 * (sec["ESTAB_county"] - sec["ESTAB_state"]) / sec["ESTAB_state"]
sec["ratio"] = sec["emp_gap_pct"] / sec["estab_gap_pct"]

view = sec.sort_values("emp_gap_pct")[
    ["EMP_state", "emp_gap", "emp_gap_pct", "estab_gap_pct", "ratio", "suppressed_cells"]
]
print(view.to_string(formatters={
    "EMP_state": "{:,.0f}".format, "emp_gap": "{:+,.0f}".format,
    "emp_gap_pct": "{:+.2f}".format, "estab_gap_pct": "{:+.2f}".format,
    "ratio": "{:6.2f}".format}))

total_gap = float(sec["emp_gap"].sum())
worst = sec["emp_gap_pct"].idxmin()

# Does a low ratio really mark the suppression-driven sectors? Measure it rather
# than assert it from the ordering of the table.
rho_sup, rho_sup_p = stats.spearmanr(sec["suppressed_cells"], sec["ratio"])
print()
print(f"total employment deficit across sectors : {total_gap:+,.0f}")
print(f"sector with the largest relative deficit: NAICS {worst}")
print(f"  its share of the total deficit        : "
      f"{100 * sec.loc[worst, 'emp_gap'] / total_gap:.1f}%")
print(f"  its employment gap                    : {sec.loc[worst, 'emp_gap_pct']:+.2f}%")
print(f"  its establishment gap                 : {sec.loc[worst, 'estab_gap_pct']:+.2f}%")
print(f"  ratio                                 : {sec.loc[worst, 'ratio']:.1f}x")
print(f"  suppressed cells in that sector       : "
      f"{int(sec.loc[worst, 'suppressed_cells']):,} of "
      f"{int((county_sectors['NAICS'] == worst).sum()):,} county rows")
print()
print(f"Spearman rho(suppressed cells, gap ratio) across {len(sec)} sectors: "
      f"{rho_sup:+.3f}  (p={rho_sup_p:.3f})")
print(f"  sectors with ratio below 1.0 : "
      f"{sorted(sec.index[sec['ratio'] < 1.0])}")
print(f"  median suppressed cells, ratio < 1.0 : "
      f"{sec.loc[sec['ratio'] < 1.0, 'suppressed_cells'].median():,.0f}")
print(f"  median suppressed cells, ratio > 2.0 : "
      f"{sec.loc[sec['ratio'] > 2.0, 'suppressed_cells'].median():,.0f}")

       EMP_state    emp_gap emp_gap_pct estab_gap_pct  ratio  suppressed_cells
NAICS                                                                         
56    12,106,284 -4,386,101      -36.23         -2.08  17.40               277
21       503,004    -89,892      -17.87        -10.56   1.69              1121
22       647,149    -66,536      -10.28        -14.06   0.73              1260
52     6,799,880   -502,732       -7.39         -2.12   3.49               105
11       168,198     -9,010       -5.36         -6.67   0.80               896
54    10,055,732   -457,171       -4.55         -0.82   5.53               126
51     3,537,983   -122,901       -3.47         -3.66   0.95               498
42     6,121,165   -184,078       -3.01         -1.43   2.11               183
55     3,634,268    -59,964       -1.65         -4.08   0.40               923
61     3,709,456    -58,473       -1.58         -1.72   0.91               735
48-49  6,345,920    -62,072       -0.98         -0.3

The prediction holds, and the two mechanisms separate cleanly.

Sectors whose ratio is near one lose establishments and employment together, which is what
removing whole rows does. That reading is not left to the eye: the rank correlation between a
sector's suppressed-cell count and its gap ratio is **-0.711 (p = 0.001)** across the
nineteen sectors, and the seven sectors with a ratio below one carry a median of 735
suppressed county cells against 105 for the sectors above two.

The sector at the top of the table behaves nothing like that. It
loses a large fraction of its employment while losing almost none of its establishments,
which is the signature of a small number of very large records held out of the county
tabulation — the employee-leasing establishments the codebook names.

For an analyst the consequence is concrete and unpleasant. Build county employment shares
from BDS, and the sector containing professional employer organisations is understated by
more than a third of itself, silently, with no flag, in a table whose every other column is
exact. A state-level analysis of the same question has no such problem.

## 4.1 The cell that is published as zero

There is one more thing to see before leaving suppression, and it is the trap this corpus
has already met in QCEW (handbook §1.4). A withheld BDS cell is not served as an empty
string or a null. It is served as the number `0`, and the only thing distinguishing it from
a genuine zero is a companion column you have to know to ask for.

In [10]:
"""What a suppressed cell looks like with and without its flag."""
suppressed_rows = county_sectors[county_sectors["EMP_F"].notna()]
example = suppressed_rows.iloc[[0]]
print("as this client returns it, with the flag requested and honoured:")
print(example[["state", "county", "NAICS", "ESTAB", "ESTAB_F", "EMP", "EMP_F"]]
      .to_string(index=False))

naive, _ = fetch_bds(
    year=YEAR, variables=["ESTAB", "EMP"],
    geography=f"county:{example['county'].iloc[0]}",
    within=f"state:{example['state'].iloc[0]}",
    naics=str(example["NAICS"].iloc[0]),
    include_flags=False,
)
print()
print("the same cell with include_flags=False, which is what a hand-rolled request gets:")
print(naive[["ESTAB", "EMP", "NAICS", "state", "county"]].to_string(index=False))

kept = county_sectors["EMP"].notna()
print()
print(f"county x sector cells whose EMP is NaN here      : {int((~kept).sum()):,}")
print(f"the same cells if the flag is not requested       : "
      f"{int((~kept).sum()):,} zeros, indistinguishable from real ones")
print(f"county x sector cells with a genuine EMP of zero : "
      f"{int((county_sectors['EMP'] == 0).sum()):,}")

as this client returns it, with the flag requested and honoured:
state county NAICS  ESTAB ESTAB_F  EMP EMP_F
   19    011    22    NaN       D  NaN     D



the same cell with include_flags=False, which is what a hand-rolled request gets:
 ESTAB  EMP NAICS state county
     0    0    22    19    011

county x sector cells whose EMP is NaN here      : 7,568
the same cells if the flag is not requested       : 7,568 zeros, indistinguishable from real ones
county x sector cells with a genuine EMP of zero : 4,244


The sums do not change — pandas skips NaN, and a zero contributes nothing to a total either
way. **What changes is whether you know.** Without the flag the frame says there are several
thousand county-sector pairs with no employment; with it, the frame says those pairs are
unmeasured. The first supports a sentence about economic structure and is false. The second
supports a sentence about data availability and is true.

## 5. The denominator that is not the denominator

BDS publishes a variable called `DENOM`. It is documented in the codebook as the
"Davis-Haltiwanger-Schuh (DHS) denominator... the average of **employment** for times t and
t-1". The name invites use as *the* denominator, and it is the wrong one for every rate that
counts establishments rather than jobs.

The FAQ gives the right construction for the establishment entry rate, and it includes an
instruction that is easy to read past: derive the prior-year establishment count *inside the
current vintage*, as `estabs_t + estabs_exit_t − estabs_entry_t`, rather than reading it
from last year's published file.

In [11]:
"""Reconstruct a published rate three ways."""
published = va_counties["ESTABS_ENTRY_RATE"]
entries = va_counties["ESTABS_ENTRY"]

prior_within_vintage = bds_prior_year_establishments(va_counties)
candidates = {
    "DENOM, as published": va_counties["DENOM"],
    "current-year ESTAB alone": va_counties["ESTAB"],
    "DHS average, prior year derived in-vintage": 0.5 * (va_counties["ESTAB"] + prior_within_vintage),
}

print(f"Virginia, {len(va_counties)} county equivalents, {YEAR}")
for label, denominator in candidates.items():
    reconstructed = 100 * entries / denominator.where(denominator > 0)
    error = (reconstructed - published).abs()
    print(f"  {label:44s} median |error| = {error.median():8.4f} pp   "
          f"reproduced = {int((error < 0.005).sum()):>3}/{len(published)}")

Virginia, 133 county equivalents, 2022
  DENOM, as published                          median |error| =   9.3369 pp   reproduced =   0/133
  current-year ESTAB alone                     median |error| =   0.0832 pp   reproduced =   5/133
  DHS average, prior year derived in-vintage   median |error| =   0.0003 pp   reproduced = 133/133


In [12]:
"""Why the vintage instruction matters: the two routes disagree about the past."""
previous_vintage, _ = fetch_bds(
    year=YEAR - 1, variables=["ESTAB"], geography="county:*", within=f"state:{FOCUS_STATE}"
)
joined = va_counties[["county", "ESTAB", "ESTABS_ENTRY", "ESTABS_EXIT"]].merge(
    previous_vintage[["county", "ESTAB"]], on="county", suffixes=("", "_prior_file")
)
joined["prior_in_vintage"] = bds_prior_year_establishments(joined)
disagree = joined["prior_in_vintage"] != joined["ESTAB_prior_file"]

print(f"counties matched across the two vintages : {len(joined)}")
print(f"counties where the two routes disagree   : {int(disagree.sum())} "
      f"({disagree.mean():.1%})")
print(f"largest absolute disagreement            : "
      f"{(joined['prior_in_vintage'] - joined['ESTAB_prior_file']).abs().max():,.0f} establishments")
print()
print("the five largest disagreements:")
worst_rows = (
    joined.assign(diff=joined["prior_in_vintage"] - joined["ESTAB_prior_file"])
    .reindex(
        (joined["prior_in_vintage"] - joined["ESTAB_prior_file"]).abs()
        .sort_values(ascending=False).index
    )
    .head(5)
)
print(worst_rows[["county", "ESTAB", "prior_in_vintage", "ESTAB_prior_file", "diff"]]
      .to_string(index=False, formatters={
          "ESTAB": "{:,.0f}".format, "prior_in_vintage": "{:,.0f}".format,
          "ESTAB_prior_file": "{:,.0f}".format, "diff": "{:+,.0f}".format}))

counties matched across the two vintages : 133
counties where the two routes disagree   : 27 (20.3%)
largest absolute disagreement            : 6 establishments

the five largest disagreements:
county  ESTAB prior_in_vintage ESTAB_prior_file diff
   087  8,769            8,544            8,550   -6
   107 10,572           10,346           10,350   -4
   059 28,767           28,709           28,707   +2
   061  1,824            1,810            1,812   -2
   810 10,503           10,396           10,398   -2


The two routes are not the same number. Roughly a fifth of Virginia's counties carry a
different prior-year establishment count depending on which file supplies it, because the
2022 vintage revised the 2021 figures it rests on.

Then the notebook's own rule applies to itself: rank the errors by size, not by how
interesting they are. The wrong denominator costs **9.34 percentage points** on a rate whose
median value is single-digit — it is not an error, it is a different statistic. Mixing
vintages costs at most a few establishments out of thousands, and moves no published rate by
more than a few hundredths of a point at this geography. Both are wrong; only one of them
would change a conclusion.

The reason to follow the FAQ's construction anyway is that its cost is zero and its error is
bounded by nothing in particular. Revisions are not guaranteed to stay this small, a series
assembled across many vintages accumulates them, and there is no diagnostic in the published
data that would tell you when they stopped being negligible. This is handbook §1.3 —
revision — arriving inside a single product rather than across releases of one.

In [13]:
"""FINDINGS"""
findings = []

findings.append((
    "F1",
    f"The published vintage shows no coarsening fingerprint: {len(emp):,} county "
    f"employment values, chi-square {chi2:.2f} on 9 df, p={p_value:.3f}, "
    f"{(digits == 0).mean():.1%} ending in zero against {0.1:.1%} expected. "
    f"Consistent with multiplicative noise infusion. Under DAO 216-26 the same three "
    f"lines read differently if coarsening replaces it, and unchanged if suppression does."
))
findings.append((
    "F2",
    f"Noise is confined to the variables Census names, and is small: across "
    f"{len(sectors)} NAICS sectors in one state the count variables sum exactly "
    f"(max |gap| {exact.abs().max():,.0f}) while the job-flow variables miss by at most "
    f"{noisy.max():.4f}%."
))
findings.append((
    "F3",
    f"Balanced noise preserves additivity, so additivity cannot detect it. Nation "
    f"versus the sum of {len(states)} states: ESTAB gap "
    f"{100 * (state_sums['ESTAB'] - national['ESTAB']) / national['ESTAB']:+.3f}%, "
    f"EMP gap {100 * (state_sums['EMP'] - national['EMP']) / national['EMP']:+.3f}%."
))
findings.append((
    "F4",
    f"Repeating an identical request {len(signatures)} times returned "
    f"{len(set(signatures))} distinct response(s): the perturbation is fixed at "
    f"tabulation and cannot be averaged away."
))
findings.append((
    "F5",
    f"FIRM is not additive over any dimension. Sum of states exceeds the national "
    f"count by {100 * (state_sums['FIRM'] - national['FIRM']) / national['FIRM']:+.2f}%; "
    f"summing counties to the state overstates firms in {int((multi['FIRM_gap_pct'] > 0).sum())} "
    f"of {len(multi)} states, median {multi['FIRM_gap_pct'].median():+.2f}%."
))
findings.append((
    "F6",
    f"The firm double-count scales with how finely the geography is cut: Spearman "
    f"rho {rho:+.3f} against the number of counties (p={rho_p:.1e}), {rho_density:+.3f} "
    f"against establishments per county. Median gap {fine.median():+.2f}% in states with "
    f">= 90 counties against {coarse.median():+.2f}% in states with <= 20."
))
findings.append((
    "F7",
    f"Counties fall short of the state on the count variables in "
    f"{int((multi['ESTAB_gap_pct'] < 0).sum())} of {len(multi)} states "
    f"(sign test p={sign_p:.1e}), median {multi['ESTAB_gap_pct'].median():+.2f}% of "
    f"establishments and {multi['EMP_gap_pct'].median():+.2f}% of employment. One-sided, "
    f"so not perturbation: the statewide (999) records are not served by the API."
))
findings.append((
    "F8",
    f"The gap ratio separates exclusion from suppression. NAICS {worst} loses "
    f"{sec.loc[worst, 'emp_gap_pct']:+.2f}% of employment against "
    f"{sec.loc[worst, 'estab_gap_pct']:+.2f}% of establishments, a ratio of "
    f"{sec.loc[worst, 'ratio']:.1f}x, and accounts for "
    f"{100 * sec.loc[worst, 'emp_gap'] / total_gap:.1f}% of the national deficit."
))
findings.append((
    "F8b",
    f"The two mechanisms are separable across sectors, not only in the extreme case: "
    f"Spearman rho between a sector's suppressed-cell count and its gap ratio is "
    f"{rho_sup:+.3f} (p={rho_sup_p:.3f}) across {len(sec)} sectors. Median suppressed "
    f"cells is {sec.loc[sec['ratio'] < 1.0, 'suppressed_cells'].median():,.0f} where the "
    f"ratio is below 1.0 and {sec.loc[sec['ratio'] > 2.0, 'suppressed_cells'].median():,.0f} "
    f"where it is above 2.0."
))
findings.append((
    "F9",
    f"Suppression is a fine-cell phenomenon: {int(flagged.sum()):,} of "
    f"{len(county_sectors):,} county-by-sector cells carry flag D "
    f"({flagged.mean():.2%}), against "
    f"{int(state_sectors['EMP_F'].notna().sum())} of {len(state_sectors):,} at state level. "
    f"Every one arrives as the number zero unless the flag column is requested."
))
findings.append((
    "F10",
    f"DENOM is the wrong denominator for establishment rates: median error "
    f"{(100 * entries / va_counties['DENOM'].where(va_counties['DENOM'] > 0) - published).abs().median():.2f} "
    f"percentage points across {len(va_counties)} counties. The in-vintage DHS average "
    f"reproduces {int(((100 * entries / (0.5 * (va_counties['ESTAB'] + prior_within_vintage))) - published).abs().lt(0.005).sum())} "
    f"of {len(published)} published rates exactly."
))
findings.append((
    "F11",
    f"Deriving the prior-year count from the previous published file instead of "
    f"in-vintage disagrees for {int(disagree.sum())} of {len(joined)} Virginia counties "
    f"({disagree.mean():.1%}), by up to "
    f"{(joined['prior_in_vintage'] - joined['ESTAB_prior_file']).abs().max():,.0f} establishments."
))

for tag, text in findings:
    print(f"[{tag}] {text}")
    print()

[F1] The published vintage shows no coarsening fingerprint: 3,142 county employment values, chi-square 7.47 on 9 df, p=0.589, 10.9% ending in zero against 10.0% expected. Consistent with multiplicative noise infusion. Under DAO 216-26 the same three lines read differently if coarsening replaces it, and unchanged if suppression does.

[F2] Noise is confined to the variables Census names, and is small: across 19 NAICS sectors in one state the count variables sum exactly (max |gap| 0) while the job-flow variables miss by at most 0.0302%.

[F3] Balanced noise preserves additivity, so additivity cannot detect it. Nation versus the sum of 51 states: ESTAB gap +0.000%, EMP gap +0.000%.

[F4] Repeating an identical request 3 times returned 1 distinct response(s): the perturbation is fixed at tabulation and cannot be averaged away.

[F5] FIRM is not additive over any dimension. Sum of states exceeds the national count by +3.73%; summing counties to the state overstates firms in 50 of 50 states,

## Key Takeaways

1. **Three mechanisms break aggregation here, and they are not the same size.** Definitional
   double counting of firms is the largest, running to double-digit percentages. Coverage
   exclusion of the statewide records is next. Injected noise is smaller than both by orders
   of magnitude and is confined to the job-flow variables. Ranking them by the attention they
   usually receive gives almost the reverse order.

2. **Additivity is not a test for perturbation.** Balanced multiplicative noise is applied
   to records before tabulation, so every sum of those records is internally consistent. A
   table that adds up tells you nothing about whether it was perturbed.

3. **The digit distribution is a test, and it survives the regime change.** Noise infusion
   leaves last digits flat; coarsening does not. The cell in §1 is the part of this notebook
   most worth keeping, because it answers a question about a *future* file.

4. **A disclosure-avoidance method has an issue date.** DAO 216-26 prohibits noise infusion
   from 4 June 2026 and makes coarsening preferred. Any BDS series that spans the transition
   will be protected two different ways with two different error structures, and the values
   will not say so. Record the regime in provenance, as this client does, or the information
   is gone.

5. **Ask for the flag column.** A withheld BDS cell is published as the number zero. The
   totals are unaffected; the sentences you are entitled to write are not.

6. **Use ESTAB when aggregating geography.** FIRM is publishable only at the geography it was
   published for, and the error from ignoring that grows with how finely you cut.

7. **Build longitudinal quantities inside one vintage.** The FAQ's derivation of the
   prior-year establishment count is not a convenience; substituting the previous published
   file reintroduces revision as measurement error.

## Exercises

1. **Re-run §1 on an older vintage.** BDS covers 1978–2023, and noise infusion entered with
   the 2007 CBP release. Does the digit test distinguish a pre-2007 tabulation from a
   post-2007 one, or was the earlier protection also invisible to it? State what a null
   result would and would not license you to conclude.

2. **Bound the statewide deficit.** §4 attributes the employment shortfall to records the API
   does not serve, and measures it only as a residual. Using the establishment gap and the
   sector ratio, construct an interval for how much employment sits in the statewide records
   of a state of your choice — and say explicitly which quantity in your interval is
   measured and which is assumed.

3. **Find the geography where FIRM is safe.** For a metropolitan area of your choosing,
   compare the published CBSA firm count against the sum of its constituent counties. Does
   the double-count vanish, shrink, or persist? What does the answer imply about using BDS
   firm counts for any geography Census did not publish?

4. **Make the suppression bias visible in a real statistic.** Compute county-level employment
   shares by sector twice — once treating flagged cells as zero, once as missing — and report
   which counties change rank. The totals are identical; the conclusions may not be.

5. **Design the successor test.** DAO 216-26 permits coarsening and, as a last resort,
   suppression. Write the diagnostic you would run on the first post-transition vintage to
   detect *suppression* replacing noise, given that §1's digit test cannot see it.

## References

- U.S. Census Bureau. *Business Dynamics Statistics (BDS) Time Series (1978–2023)*, API
  documentation. `api.census.gov/data/timeseries/bds`. Disclosure Review Board approval
  CBDRB-FY25-0331.
- U.S. Census Bureau. *BDS Codebook and Glossary (2023 Release)* — flag values `D`, `S`, `X`,
  `N`; the statewide (999) county code; the DHS denominator.
- U.S. Census Bureau. *BDS Frequently Asked Questions* — firm counts across geography and
  industry; the entry-rate denominator and its in-vintage derivation; the list of
  noise-infused variables.
- U.S. Census Bureau. *BDS Methodology* — hybrid balanced multiplicative noise infusion, and
  the standing notice that the description is no longer current.
- U.S. Department of Commerce. *Department Administrative Order 216-26, "Disclosure Avoidance
  for Statistical Products"*, effective 4 June 2026. §5.01(c) prohibits noise infusion;
  §5.02 makes coarsening preferred and permits suppression as a last resort.
- Cook, G. M. *Understanding the New Disclosure Avoidance Policy: What It Means for Data,
  Confidentiality and You*. U.S. Census Bureau Director's Blog, 17 August 2026.
- Davis, S. J., Haltiwanger, J. C., and Schuh, S. *Job Creation and Destruction*. MIT Press,
  1996 — the denominator convention BDS rate variables use.
- Evans, T., Zayatz, L., and Slanta, J. "Using Noise for Disclosure Limitation of
  Establishment Tabular Data." *Journal of Official Statistics* 14(4), 1998 — the
  multiplicative noise method BDS adapted from County Business Patterns.

## Limitations

- **One reference year.** Every measurement is 2022. The three mechanisms are structural and
  should be stable, but their magnitudes are not measured across years here, and the
  statewide-records deficit in particular could move with the size of the leasing industry.
- **The statewide (999) records are inferred, never observed.** The API does not serve them.
  §4 establishes that the deficit behaves exactly as the codebook's description predicts, and
  that a suppression-only explanation does not fit the sector pattern. It does not observe a
  single statewide record, and a reader who needs the actual figure must go to the published
  flat files rather than the API.
- **The sector attribution is a pattern match, not a decomposition.** The employment gap in
  each sector is a residual containing both mechanisms. The ratio separates which one
  dominates; it does not apportion the gap between them.
- **The digit test detects rounding, not suppression.** It would pass unchanged on a file
  protected entirely by withholding, which DAO 216-26 permits. Exercise 5 exists because this
  notebook does not solve that.
- **The one-state sections generalise only as far as they were checked.** §5's rate
  reconstruction and the vintage comparison use Virginia's 133 county equivalents. §3 and §4
  are national.
- **Noise magnitude is measured at state and national geography.** At county-by-sector cells
  the perturbation is proportionally larger than the tens-of-jobs figures in §2, and nothing
  here bounds it there.
- **Not tested here:** whether the published rate variables are computed before or after the
  noise is applied to their components, which determines whether a reconstructed rate and a
  published rate should be expected to agree at fine geography at all.